**Homesite_Assignment_Harshal_Samuel_Albert**

Importing Libraries & G-Drive Set Up


In [1]:
  # To upload our datasets from our working directory we need to mount our drive contents to the colab environment.
  # For the code to do so you can search “mount” in code snippets or use the code given below.
  # Our entire drive contents are now mounted on colab at the location “/gdrive”.
  from google.colab import drive
  drive.mount('/gdrive')
  #Change current working directory to gdrive
  %cd /gdrive


Mounted at /gdrive
/gdrive


In [2]:
!pip install vecstack

  Preparing metadata (setup.py) ... done
  Created wheel for vecstack: filename=vecstack-0.4.0-py3-none-any.whl size=19861 sha256=d6c7bb2f76f01a14cd476cb74a1e6abff35c233445bfbdddc207cd5cd3b41493
  Stored in directory: /root/.cache/pip/wheels/b8/d8/51/3cf39adf22c522b0a91dc2208db4e9de4d2d9d171683596220
Successfully built vecstack


In [3]:
!pip install --upgrade scikit-learn

In [4]:
#from sklearn.datasets.samples_generator import make_blobs #This module is deprecated
from sklearn.datasets import make_blobs #make_blobs has been moved to sklearn.datasets
import pandas as pd
import numpy as np
from sklearn.metrics import accuracy_score #works
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import RandomizedSearchCV
from sklearn.model_selection import cross_val_score
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.ensemble import GradientBoostingClassifier
#from sklearn.datasets.samples_generator import make_blobs
from sklearn.neighbors import KNeighborsClassifier
from matplotlib import pyplot as plt
from sklearn.calibration import CalibratedClassifierCV
from sklearn.neural_network import MLPClassifier
#from sklearn.ensemble import RandomForestClassifier
from imblearn.over_sampling import SMOTE
from sklearn.svm import SVC
from sklearn import svm
from collections import Counter #for Smote,

import warnings
warnings.filterwarnings("ignore")

Exploratory Data Analysis


In [5]:

trainfile = r'/content/RevisedHomesiteTrain1.csv'
train_data = pd.read_csv(trainfile)

#train_data


testfile = r'/content/RevisedHomesiteTest1.csv'
test_data = pd.read_csv(testfile)

#test_data


print(train_data.shape)
print(test_data.shape)

(65000, 596)
(173836, 596)


In [6]:
test_data.head()

,CoverageField11A,CoverageField11B,CoverageField1A,CoverageField1B,CoverageField2A,CoverageField2B,CoverageField3A,CoverageField3B,CoverageField4A,CoverageField4B,...,PropertyField38_N,PropertyField38_Y,GeographicField63_,GeographicField63_N,GeographicField63_Y,GeographicField64_CA,GeographicField64_IL,GeographicField64_NJ,GeographicField64_TX,GeographicField64
0,13,22,4,4,4,4,3,3,3,4,...,1,0,0,0,1,0,0,0,0,IL
1,4,5,8,14,8,14,7,12,8,13,...,1,0,0,1,0,0,0,0,0,NJ
2,3,3,11,18,11,18,10,16,10,18,...,1,0,0,1,0,0,0,0,0,NJ
3,5,9,14,22,15,22,13,20,22,25,...,1,0,0,1,0,0,0,0,0,TX
4,12,21,4,5,4,5,4,4,4,5,...,1,0,0,1,0,0,0,0,0,CA


In [7]:
test_data.columns

Index(['CoverageField11A', 'CoverageField11B', 'CoverageField1A',
       'CoverageField1B', 'CoverageField2A', 'CoverageField2B',
       'CoverageField3A', 'CoverageField3B', 'CoverageField4A',
       'CoverageField4B',
       ...
       'PropertyField38_N', 'PropertyField38_Y', 'GeographicField63_ ',
       'GeographicField63_N', 'GeographicField63_Y', 'GeographicField64_CA',
       'GeographicField64_IL', 'GeographicField64_NJ', 'GeographicField64_TX',
       'GeographicField64'],
      dtype='object', length=596)

In [8]:
test_data.isnull().sum()

,0
CoverageField11A,0
CoverageField11B,0
CoverageField1A,0
CoverageField1B,0
CoverageField2A,0
...,...
GeographicField64_CA,0
GeographicField64_IL,0
GeographicField64_NJ,0
GeographicField64_TX,0


In [9]:
#Drop categorical features without encoding
categoricalFeatures = ['GeographicField64_CA',
'GeographicField64_IL', 'GeographicField64_NJ',
'GeographicField64_TX']
subData = test_data.drop(columns = categoricalFeatures)

#Do one-hot encoding
dV = pd.get_dummies(test_data['GeographicField64'],
prefix= 'GeographicField64')
dV

#Dataframe Conversion
test_data = pd.concat([subData, dV], axis=1)
test_data = test_data.drop(columns='GeographicField64')

X_train = train_data.iloc[:, :-1].copy()
X_test = test_data.copy()
Y_train = train_data.iloc[:, -1].copy()

Models

Decision Tree Classifier

In [10]:
#Create Decision Tree
clf = DecisionTreeClassifier()
clf.fit(X_train, Y_train)

#Check accuracy on training set
clf_pred_train = clf.predict(X_train)
print('TRAIN ACCURACY: ', accuracy_score(Y_train, clf_pred_train),'\n')
#Check accuracy on test set
clf_pred = clf.predict(X_test)
print('Confusion Matrix: \n', confusion_matrix(Y_train, clf_pred_train),'\n')
print('Classification Report: \n', classification_report(Y_train, clf_pred_train))

TRAIN ACCURACY:  1.0 

Confusion Matrix: 
 [[52738     0]
 [    0 12262]] 

Classification Report: 
               precision    recall  f1-score   support

           0       1.00      1.00      1.00     52738
           1       1.00      1.00      1.00     12262

    accuracy                           1.00     65000
   macro avg       1.00      1.00      1.00     65000
weighted avg       1.00      1.00      1.00     65000



Hyperparameter Tuning for Decision Tree Classifier

In [11]:
#Hyperparameter tuning done for decision tree classifier
parameters={'min_samples_split' : range(10,100,10),'max_depth': range(1,20,2)}
clf_random = RandomizedSearchCV(clf,parameters,n_iter=15)
clf_random.fit(X_train, Y_train)
grid_parm=clf_random.best_params_
print(grid_parm)

#Using the parameters obtained from HyperParameterTuning in the DecisionTreeClassifier
clf = DecisionTreeClassifier(**grid_parm)
clf.fit(X_train,Y_train)
clf_predict = clf.predict(X_test)

#Obtain accuracy ,confusion matrix,classification report and AUC values for the result above.
print("accuracy Score (train) after hypertuning for Decision Tree:{0:6f}".format(clf.score(X_train,Y_train)))

#get cross-validation report
clf_cv_score = cross_val_score(clf, X_train, Y_train, cv=10, scoring="roc_auc")
print("=== All AUC Scores ===")
print(clf_cv_score)
print('\n')
print("=== Mean AUC Score ===")
print("Mean AUC Score - Decision Tree: ",clf_cv_score.mean())

{'min_samples_split': 60, 'max_depth': 9}
accuracy Score (train) after hypertuning for Decision Tree:0.919385
=== All AUC Scores ===
[0.94577782 0.9432191  0.9377693  0.9441001  0.9398352  0.94811198
 0.94780568 0.93428573 0.94769211 0.92846908]


=== Mean AUC Score ===
Mean AUC Score - Decision Tree:  0.9417066095372943


In [12]:
#Get Prediction Probability for the predicted class as a dataframe
clf_Probability = pd.DataFrame(clf.predict_proba(X_test))
clf_Probability.head()

,0,1
0,1.000000,0.000000
1,0.921053,0.078947
2,0.963710,0.036290
3,0.963710,0.036290
4,0.927978,0.072022


from matplotlib import pyplot as plt
_df_0[0].plot(kind='hist', bins=20, title=0)
plt.gca().spines[['top', 'right',]].set_visible(False)

from matplotlib import pyplot as plt
_df_1[1].plot(kind='hist', bins=20, title=1)
plt.gca().spines[['top', 'right',]].set_visible(False)

from matplotlib import pyplot as plt
_df_2.plot(kind='scatter', x=0, y=1, s=32, alpha=.8)
plt.gca().spines[['top', 'right',]].set_visible(False)

from matplotlib import pyplot as plt
_df_3[0].plot(kind='line', figsize=(8, 4), title=0)
plt.gca().spines[['top', 'right']].set_visible(False)

from matplotlib import pyplot as plt
_df_4[1].plot(kind='line', figsize=(8, 4), title=1)
plt.gca().spines[['top', 'right']].set_visible(False)

Random Forest Classifier

In [13]:
#Create Random Forest
rfc = RandomForestClassifier()
rfc.fit(X_train, Y_train)

#Check accuracy on training set
rfc_pred_train = rfc.predict(X_train)
print('TRAIN ACCURACY: ', accuracy_score(Y_train, rfc_pred_train),'\n')
#Check accuracy on test set
rfc_pred = rfc.predict(X_test)
print('Confusion Matrix: \n', confusion_matrix(Y_train, rfc_pred_train),'\n')
print('Classification Report: \n', classification_report(Y_train, rfc_pred_train))

TRAIN ACCURACY:  1.0 

Confusion Matrix: 
 [[52738     0]
 [    0 12262]] 

Classification Report: 
               precision    recall  f1-score   support

           0       1.00      1.00      1.00     52738
           1       1.00      1.00      1.00     12262

    accuracy                           1.00     65000
   macro avg       1.00      1.00      1.00     65000
weighted avg       1.00      1.00      1.00     65000



Hyperparameter Tuning for Random Forest Classifier

In [14]:
from sklearn.model_selection import RandomizedSearchCV, cross_val_score
from sklearn.ensemble import RandomForestClassifier

# Narrowed parameter range to reduce computation time
parameters = {
    'min_samples_split': [2, 5, 10],  # Smaller range of values
    'max_depth': [5, 10, 15]         # Reduced range for max_depth
}

# Enable parallel processing with n_jobs=-1
rfc_random = RandomizedSearchCV(
    RandomForestClassifier(),
    parameters,
    n_iter=10,  # Reduced number of iterations
    n_jobs=-1,  # Parallel computation to use all available processors
    cv=5,       # Using 5-fold cross-validation
    scoring="roc_auc",  # Focus on AUC scoring
    verbose=2           # Add verbosity to track progress
)

# Fit RandomizedSearchCV
rfc_random.fit(X_train, Y_train)

# Get the best parameters
grid_param_rfc = rfc_random.best_params_
print("Best Parameters:", grid_param_rfc)

# Validate the format and structure of the best parameters
print(type(grid_param_rfc))  # Should output: <class 'dict'>
print(grid_param_rfc)        # To ensure it contains the expected key-value pairs

# Using the parameters obtained from hyperparameter tuning
rfc = RandomForestClassifier(**grid_param_rfc)

# Fit the model with the optimized parameters
rfc.fit(X_train, Y_train)

# Predict on the test set
rfc_predict = rfc.predict(X_test)

# Accuracy score after hyperparameter tuning
print("Accuracy Score (train) after hyperparameter tuning for Random Forest: {:.6f}".format(rfc.score(X_train, Y_train)))

# Cross-validation report
rfc_cv_score = cross_val_score(
    rfc, X_train, Y_train, cv=5, scoring="roc_auc", n_jobs=-1  # Parallelized CV
)
print("\n=== All AUC Scores ===")
print(rfc_cv_score)
print("\n=== Mean AUC Score ===")
print("Mean AUC Score - Random Forest:", rfc_cv_score.mean())


Fitting 5 folds for each of 9 candidates, totalling 45 fits
Best Parameters: {'min_samples_split': 2, 'max_depth': 15}
<class 'dict'>
{'min_samples_split': 2, 'max_depth': 15}
Accuracy Score (train) after hyperparameter tuning for Random Forest: 0.950446

=== All AUC Scores ===
[0.94733247 0.94412376 0.9490695  0.94707417 0.9443572 ]

=== Mean AUC Score ===
Mean AUC Score - Random Forest: 0.9463914195642396


In [15]:
#Get Prediction Probability for the predicted class as a dataframe
rfc_Probability = pd.DataFrame(rfc.predict_proba(X_test))
rfc_Probability.head()

,0,1
0,0.972192,0.027808
1,0.902151,0.097849
2,0.918792,0.081208
3,0.979890,0.020110
4,0.767224,0.232776


Support Vector Machines Classifier

In [16]:
svc = svm.LinearSVC()
svc.fit(X_train, Y_train)
svc_pred_train=svc.predict(X_train)
print('TRAIN ACCURACY: ', accuracy_score(Y_train, svc_pred_train),'\n')
#Check accuracy on test set
svc_pred = svc.predict(X_test)
print('Confusion Matrix: \n', confusion_matrix(Y_train, svc_pred_train),'\n')
print('Classification Report: \n', classification_report(Y_train, svc_pred_train))

TRAIN ACCURACY:  0.8288153846153846 

Confusion Matrix: 
 [[51895   843]
 [10284  1978]] 

Classification Report: 
               precision    recall  f1-score   support

           0       0.83      0.98      0.90     52738
           1       0.70      0.16      0.26     12262

    accuracy                           0.83     65000
   macro avg       0.77      0.57      0.58     65000
weighted avg       0.81      0.83      0.78     65000



Hyperparameter Tuning for Support Vector Machines

In [17]:
#Hyperparameter tuning done for support vector machines classifier
parameters={'C' : [0.8, 5, 20]}
svc_random = RandomizedSearchCV(svc,parameters,n_iter=15)
svc_random.fit(X_train, Y_train)
grid_parm_svc=svc_random.best_params_
print(grid_parm_svc)

#Using the parameters obtained from HyperParameterTuning in the SupportVectorMachines
svc = svm.LinearSVC(**grid_parm_svc)
svc.fit(X_train,Y_train)
svc_predict = svc.predict(X_test)

#Obtain accuracy ,confusion matrix,classification report and AUC values for the result above.
print("accuracy Score (train) after hypertuning for SVM:{0:6f}".format(svc.score(X_train,Y_train)))

#get cross-validation report
svc_cv_score = cross_val_score(svc, X_train, Y_train, cv=10, scoring="roc_auc")
print("=== All AUC Scores ===")
print(svc_cv_score)
print('\n')
print("=== Mean AUC Score ===")
print("Mean AUC Score - SVM: ",svc_cv_score.mean())

{'C': 5}
accuracy Score (train) after hypertuning for SVM:0.827615
=== All AUC Scores ===
[0.81212043 0.84715054 0.83180409 0.93554301 0.82860733 0.88474439
 0.82918559 0.85231113 0.89451266 0.82663709]


=== Mean AUC Score ===
Mean AUC Score - SVM:  0.8542616261332867


In [18]:
# Instead of:
# svcC=CalibratedClassifierCV(base_estimator=svc, cv=4)

# Use:
!pip install scikit-learn --upgrade
from sklearn.calibration import CalibratedClassifierCV
svcC=CalibratedClassifierCV(svc, cv=4) # Pass the base estimator as the first argument
svcC.fit(X_train, Y_train)

CalibratedClassifierCV(cv=4, estimator=LinearSVC(C=5))

In [19]:
#Get Prediction Probability for the predicted class as a dataframe
svc_Probability = pd.DataFrame(svcC.predict_proba(X_test))
svc_Probability.head()

,0,1
0,0.959631,0.040369
1,0.892082,0.107918
2,0.942853,0.057147
3,0.978040,0.021960
4,0.865702,0.134298


K-Nearest Neighbors Classifier

In [20]:
knc = KNeighborsClassifier()
knc.fit(X_train, Y_train)
knc_pred_train=knc.predict(X_train)
print('TRAIN ACCURACY: ', accuracy_score(Y_train, knc_pred_train),'\n')
#Check accuracy on test set
knc_pred = knc.predict(X_test)
print('Confusion Matrix: \n', confusion_matrix(Y_train, knc_pred_train),'\n')
print('Classification Report: \n', classification_report(Y_train, knc_pred_train))

TRAIN ACCURACY:  0.8292153846153846 

Confusion Matrix: 
 [[51551  1187]
 [ 9914  2348]] 

Classification Report: 
               precision    recall  f1-score   support

           0       0.84      0.98      0.90     52738
           1       0.66      0.19      0.30     12262

    accuracy                           0.83     65000
   macro avg       0.75      0.58      0.60     65000
weighted avg       0.81      0.83      0.79     65000



###Hyperparameter Tuning for K_Nearest Neighbor Classifier

In [22]:
!pip install scikit-learn # install scikit-learn if necessary
from sklearn.model_selection import RandomizedSearchCV # Import RandomizedSearchCV

# Hyperparameter tuning done for K-Nearest Neighbor classifier
# ... (rest of your code)

In [24]:
from sklearn.model_selection import RandomizedSearchCV, cross_val_score
from sklearn.neighbors import KNeighborsClassifier

In [25]:
# Hyperparameter tuning done for K-Nearest Neighbor classifier
parameters = {
    'n_neighbors': range(10, 31, 10),  # Smaller range for faster evaluation
    'weights': ['uniform', 'distance'],  # Same options for weight
    'p': [2]  # Using only one value for p to reduce complexity (or keep [2, 4] if necessary)
}

# RandomizedSearchCV with fewer iterations and parallel processing
knc_random = RandomizedSearchCV(
    KNeighborsClassifier(),
    parameters,
    n_iter=5,          # Keep iterations low for quicker execution
    n_jobs=-1,         # Enable parallel computation
    cv=3,              # Reduce folds in cross-validation (e.g., 3 instead of 5)
    scoring="roc_auc", # Use AUC for scoring
    verbose=2          # Add verbosity to monitor progress
)

# Fit the RandomizedSearchCV
knc_random.fit(X_train, Y_train)

# Get the best parameters
grid_parm_knc = knc_random.best_params_
print("Best Parameters:", grid_parm_knc)

# Using the parameters obtained from HyperParameterTuning in the K_NearestNeighborClassifier
knc = KNeighborsClassifier(**grid_parm_knc)
knc.fit(X_train, Y_train)
knc_predict = knc.predict(X_test)

# Obtain accuracy, confusion matrix, classification report, and AUC values for the result above
print("Accuracy Score (train) after hypertuning for KNN: {:.6f}".format(knc.score(X_train, Y_train)))

# Get cross-validation report
knc_cv_score = cross_val_score(
    knc, X_train, Y_train, cv=3, scoring="roc_auc", n_jobs=-1  # Parallelized cross-validation
)
print("=== All AUC Scores ===")
print(knc_cv_score)
print("\n=== Mean AUC Score ===")
print("Mean AUC Score - KNN:", knc_cv_score.mean())


Fitting 3 folds for each of 5 candidates, totalling 15 fits
Best Parameters: {'weights': 'distance', 'p': 2, 'n_neighbors': 20}
Accuracy Score (train) after hypertuning for KNN: 1.000000
=== All AUC Scores ===
[0.50948155 0.5095247  0.50873291]

=== Mean AUC Score ===
Mean AUC Score - KNN: 0.5092463867470506


In [26]:
#Get Prediction Probability for the predicted class as a dataframe
knc_Probability = pd.DataFrame(knc.predict_proba(X_test))
knc_Probability.head()

,0,1
0,0.807474,0.192526
1,0.762469,0.237531
2,0.753927,0.246073
3,0.910898,0.089102
4,0.787648,0.212352


##MultiLayer Perceptron Classifier

In [27]:
mlp = MLPClassifier()
mlp.fit(X_train, Y_train)
mlp_pred_train=mlp.predict(X_train)
print('TRAIN ACCURACY: ', accuracy_score(Y_train, mlp_pred_train),'\n')
#Check accuracy on test set
mlp_pred = mlp.predict(X_test)
print('Confusion Matrix: \n', confusion_matrix(Y_train, mlp_pred_train),'\n')
print('Classification Report: \n', classification_report(Y_train, mlp_pred_train))

TRAIN ACCURACY:  0.8461538461538461 

Confusion Matrix: 
 [[51769   969]
 [ 9031  3231]] 

Classification Report: 
               precision    recall  f1-score   support

           0       0.85      0.98      0.91     52738
           1       0.77      0.26      0.39     12262

    accuracy                           0.85     65000
   macro avg       0.81      0.62      0.65     65000
weighted avg       0.84      0.85      0.81     65000



###Hyperparameter Tuning for MultiLayer Perceptron Classifier

In [28]:
#Hyperparameter tuning done for MultiLayer Perceptron classifier
parameters = {'hidden_layer_sizes':[(10,5,3), (20,7,3)], 'activation':['tanh', 'relu'],
              'learning_rate':['constant', 'adaptive'], 'max_iter' :[100, 150]}
mlp_random = RandomizedSearchCV(mlp,parameters,n_iter=8)
mlp_random.fit(X_train, Y_train)
grid_parm_mlp=mlp_random.best_params_
print(grid_parm_mlp)

#Using the parameters obtained from HyperParameterTuning in the MLPClassifier
mlp = MLPClassifier(**grid_parm_mlp)
mlp.fit(X_train,Y_train)
mlp_predict = mlp.predict(X_test)

#Obtain accuracy ,confusion matrix,classification report and AUC values for the result above.
print("accuracy Score (train) after hypertuning for MLP:{0:6f}".format(mlp.score(X_train,Y_train)))

#get cross-validation report
mlp_cv_score = cross_val_score(mlp, X_train, Y_train, cv=5, scoring="roc_auc")
print("=== All AUC Scores ===")
print(mlp_cv_score)
print('\n')
print("=== Mean AUC Score ===")
print("Mean AUC Score - MLP: ",mlp_cv_score.mean())

{'max_iter': 150, 'learning_rate': 'adaptive', 'hidden_layer_sizes': (20, 7, 3), 'activation': 'tanh'}
accuracy Score (train) after hypertuning for MLP:0.811354
=== All AUC Scores ===
[0.49553743 0.50762532 0.49683075 0.5        0.50108833]


=== Mean AUC Score ===
Mean AUC Score - MLP:  0.5002163665238188


In [29]:
#Get Prediction Probability for the predicted class as a dataframe
mlp_Probability = pd.DataFrame(mlp.predict_proba(X_test))
mlp_Probability.head()

,0,1
0,0.808206,0.191794
1,0.808206,0.191794
2,0.808206,0.191794
3,0.808206,0.191794
4,0.808206,0.191794


SMOTE

In [31]:
!pip install imblearn --upgrade
print("___________________________________________________________________\nSMOTE\n")
print('Original dataset shape %s' % Counter(Y_train))
# Replace 'ratio' with 'sampling_strategy' and set it to 0.5
sm = SMOTE(sampling_strategy=0.5)
X_train, Y_train = sm.fit_resample(X_train, Y_train)
print('Resampled dataset shape %s' % Counter(Y_train))

___________________________________________________________________
SMOTE

Original dataset shape Counter({0: 52738, 1: 12262})
Resampled dataset shape Counter({0: 52738, 1: 26369})


ENSEMBLE METHODS STACKING

In [33]:
!pip install mlxtend
from mlxtend.classifier import StackingClassifier
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.neural_network import MLPClassifier
from collections import Counter
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split, cross_val_score
from imblearn.over_sampling import SMOTE

print("___________________________________________________________________________________________\nEnsemble Methods Predictions using GradientBoosting, RandomForest and Decision Tree Classifier\n")

#models = [ svm.LinearSVC(), RandomForestClassifier(), DecisionTreeClassifier(), KNeighborsClassifier(), MLPClassifier() ]
models = [ DecisionTreeClassifier(min_samples_split= 70, max_depth= 9)  ,
          RandomForestClassifier(min_samples_split= 90, max_depth= 9) ,
          MLPClassifier(max_iter= 100, learning_rate= 'constant',
                        hidden_layer_sizes= (20, 7, 3), activation= 'tanh') ,
          KNeighborsClassifier(n_neighbors= 3) ,
           SVC(max_iter= 700, probability=True) ] # SVC needs probability=True for StackingClassifier

# Initialize the StackingClassifier
meta_classifier = LogisticRegression()
stacking_classifier = StackingClassifier(classifiers=models,
                                        meta_classifier=meta_classifier,
                                        use_probas=True) # Assuming you want to use probabilities

# Fit and predict using the StackingClassifier
stacking_classifier.fit(X_train, Y_train)
S_Train = stacking_classifier.predict(X_train)
S_Test = stacking_classifier.predict(X_test)

#print('Resampled dataset shape %s' % Counter(Y_train))

___________________________________________________________________________________________
Ensemble Methods Predictions using GradientBoosting, RandomForest and Decision Tree Classifier



Hyperparameter Tuning for Ensemble Methods Stacking

In [35]:
ems = GradientBoostingClassifier()
# Reshape S_Train to a 2D array
ems = ems.fit(S_Train.reshape(-1, 1), Y_train)  # Reshape S_Train to have one column
y_pred = ems.predict(S_Test.reshape(-1, 1))  # Reshape S_Test for prediction as well

In [37]:
#Hyperparameter tuning done for stacked model
parameters = {'n_estimators': [100, 200, 350], 'loss': ['log_loss', 'exponential'], # Changed 'deviance' to 'log_loss'
              'criterion': ['friedman_mse', 'squared_error']}
ems_random = RandomizedSearchCV(ems,parameters,n_iter=5)
# Reshape S_Train before fitting to RandomizedSearchCV
ems_random.fit(S_Train.reshape(-1, 1), Y_train)
grid_parm_ems=ems_random.best_params_
print(grid_parm_ems)

#Using the parameters obtained from HyperParameterTuning in the stacked model
ems = GradientBoostingClassifier(**grid_parm_ems)
# Reshape S_Train before fitting to GradientBoostingClassifier
ems.fit(S_Train.reshape(-1, 1),Y_train)
ems_predict = ems.predict(S_Test.reshape(-1, 1))  # Reshape S_Test for prediction

#Obtain accuracy ,confusion matrix,classification report and AUC values for the result above.
print("accuracy Score (train) after hypertuning for Stacked Model:{0:6f}".format(ems.score(S_Train.reshape(-1, 1),Y_train)))

#get cross-validation report
ems_cv_score = cross_val_score(ems, S_Train.reshape(-1, 1), Y_train, cv=7, scoring="roc_auc") # Reshape S_Train here as well
print("=== All AUC Scores ===")
print(ems_cv_score)
print('\n')
print("=== Mean AUC Score ===")
print("Mean AUC Score - Stacked Model: ",ems_cv_score.mean())

{'n_estimators': 350, 'loss': 'exponential', 'criterion': 'squared_error'}
accuracy Score (train) after hypertuning for Stacked Model:0.958322
=== All AUC Scores ===
[0.91345899 0.91186621 0.91465357 0.96150783 0.98506769 0.98480223
 0.98520042]


=== Mean AUC Score ===
Mean AUC Score - Stacked Model:  0.9509367059805075


In [39]:
#Get Prediction Probability for the predicted class as a dataframe
stack_Probability = pd.DataFrame(ems.predict_proba(S_Test.reshape(-1, 1))) # Reshape S_Test to a 2D array

stack_Probability.head()


,0,1
0,0.964697,0.035303
1,0.964697,0.035303
2,0.964697,0.035303
3,0.964697,0.035303
4,0.964697,0.035303


Code to Save Results as CSV

Saving Results File


In [90]:
import pandas as pd
import os
from google.colab import drive # Import the necessary module

# Mount Google Drive
drive.mount('/content/drive') # Mount to a specific directory

# Assuming `test_data` contains the `QuoteNumber` column and predictions are available
quote_numbers = test_data['QuoteNumber']  # Replace with the actual column name for unique IDs
final_predictions = clf_predict  # Replace with your model's predictions (binary classification: 0 or 1)

# Create the submission DataFrame
submission = pd.DataFrame({
    'QuoteNumber': quote_numbers,
    'QuoteConversion_Flag': final_predictions  # Ensure this contains binary predictions (0/1)
})

# Get the current working directory
current_directory = '/content/drive/My Drive'  # Change to your desired directory in Google Drive

# Define the path for the submission file
submission_file_name = os.path.join(current_directory, 'submission.csv')

# Save to CSV file
submission.to_csv(submission_file_name, index=False)
print(f"Submission file saved as '{submission_file_name}'.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Submission file saved as '/content/drive/My Drive/submission.csv'.
